# Batch convert PDFs and JPEGs to Excel

Paste your Datalab API key below, then run all cells. The notebook converts every `.pdf`, `.jpg`, and `.jpeg` in `INPUT_FOLDER`; each input gets its own Excel workbook, with one worksheet per detected table.

If you follow the README and start Jupyter from the project folder, leave the folder settings unchanged. Put source files in `data/`; Excel workbooks are written to `outputs/`.

In [ ]:
# Uncomment and run each line once if the package is missing.
# %pip install jupyterlab
# %pip install pandas
# %pip install requests
# %pip install lxml
# %pip install openpyxl

In [ ]:
from io import StringIO
from pathlib import Path
import os
import time

import pandas as pd
import requests

API_KEY = ""  # Paste your Datalab API key here for a personal local copy.
INPUT_FOLDER = Path("data")
OUTPUT_FOLDER = Path("outputs")
POLL_SECONDS = 5

# Alternatively, set DATALAB_API_KEY in your environment instead of pasting it above.
API_KEY = API_KEY or os.getenv("DATALAB_API_KEY", "")

if not API_KEY:
    raise ValueError("Paste your Datalab API key into API_KEY or set DATALAB_API_KEY.")

MIME_TYPES = {
    ".pdf": "application/pdf",
    ".jpg": "image/jpeg",
    ".jpeg": "image/jpeg",
}


In [ ]:
def find_tables(item):
    if isinstance(item, dict):
        if item.get("block_type") == "Table":
            yield item
        for value in item.values():
            yield from find_tables(value)
    elif isinstance(item, list):
        for value in item:
            yield from find_tables(value)


def flatten_columns(dataframe):
    if not isinstance(dataframe.columns, pd.MultiIndex):
        return dataframe

    dataframe = dataframe.copy()
    dataframe.columns = [
        " | ".join(
            dict.fromkeys(
                str(value) for value in column if not str(value).startswith("Unnamed")
            )
        )
        for column in dataframe.columns
    ]
    return dataframe


input_files = [
    path
    for path in INPUT_FOLDER.iterdir()
    if path.is_file() and path.suffix.lower() in MIME_TYPES
]

if not input_files:
    raise FileNotFoundError("No PDF, JPG, or JPEG files were found in INPUT_FOLDER.")

for input_path in input_files:
    print(f"Uploading {input_path.name}...")

    with input_path.open("rb") as file:
        job = requests.post(
            "https://www.datalab.to/api/v1/convert",
            headers={"X-API-Key": API_KEY},
            files={"file": (input_path.name, file, MIME_TYPES[input_path.suffix.lower()])},
            data={"output_format": "json", "mode": "accurate"},
            timeout=60,
        )
    job.raise_for_status()
    check_url = job.json()["request_check_url"]

    while True:
        response = requests.get(check_url, headers={"X-API-Key": API_KEY}, timeout=60)
        response.raise_for_status()
        result = response.json()

        if result["status"] == "complete":
            break
        if result["status"] in {"failed", "error"}:
            raise RuntimeError(f"Datalab processing failed: {result}")

        time.sleep(POLL_SECONDS)

    dataframes = [
        pd.read_html(StringIO(table["html"]))[0]
        for table in find_tables(result["json"])
        if table.get("html")
    ]

    if not dataframes:
        print(f"No tables found in {input_path.name}.")
        continue

    OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
    excel_path = OUTPUT_FOLDER / f"{input_path.stem}_tables.xlsx"
    with pd.ExcelWriter(excel_path) as writer:
        for index, dataframe in enumerate(dataframes, start=1):
            flatten_columns(dataframe).to_excel(
                writer, sheet_name=f"Table_{index}", index=False
            )

    print(f"Saved: {excel_path}")